<a href="https://colab.research.google.com/github/Yacqueline12/tecnm-lenguajes-automatas-2/blob/main/laboratorios/sesion06/sesion06_laboratorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lenguajes y Autómatas II (SCD-1016)
## Sesión 06 · Sistema de Tipos Primitivos y Verificación Semántica

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo decide el compilador si una operación entre dos tipos tiene sentido?
>
> **Objetivo de la sesión:** Aplicar una tabla de compatibilidad de tipos y explicar por qué asignar un tipo más ancho a uno más angosto exige una conversión explícita.

**Entregable:** tu tabla en artesano_sesion06_c.c, el harness en verde y la bitácora

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Implementas compatible(destino, origen) y nombre_tipo(t); el harness prueba asignaciones legales e ilegales y el nombre de cada tipo.

Al final de la sesión tendrás: **Una tabla de compatibilidad en C que acepta asignaciones seguras, rechaza las que pierden datos y nombra cada tipo.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 06](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion06.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion06.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion06.h"

int compatible(Tipo destino, Tipo origen);
const char *nombre_tipo(Tipo t);
```

- La jerarquía es CHAR < INT < FLOAT.
- Asignar a un tipo más angosto se rechaza: requiere cast explícito.
- TIPO_ERROR nunca es compatible con nada.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion06_c.c` junto con `artesano_sesion06_c.c`
> (o `ia_sesion06_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `compatible(INT,INT)` | 1 |
| 2 | `compatible(FLOAT,INT)` | 1 |
| 3 | `compatible(INT,FLOAT)` | 0 |
| 4 | `compatible(INT,ERROR)` | 0 |
| 5 | `nombre_tipo(INT)` | `int` |
| 6 | `nombre_tipo(ERROR)` | `error` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 06 - Tabla de compatibilidad de tipos ===
Caso                               esperado:              obtenido:              resultado
[ 1] compatible(INT,INT)                esperado: 1            obtenido: 1            OK
[ 2] compatible(FLOAT,INT)              esperado: 1            obtenido: 1            OK
[ 3] compatible(INT,FLOAT)              esperado: 0            obtenido: 0            OK
[ 4] compatible(INT,ERROR)              esperado: 0            obtenido: 0            OK
[ 5] nombre_tipo(INT)                   esperado: int          obtenido: int          OK
[ 6] nombre_tipo(ERROR)                 esperado: error        obtenido: error        OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion06.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] La compatibilidad solo acepta destinos de igual o mayor rango.
- [ ] TIPO_ERROR se rechaza siempre y se nombra como error.
- [ ] Los seis casos del harness salen OK con código de salida 0.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion06_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [13]:
%%writefile api_sesion06.h
/* api_sesion06.h - contrato de la Sesion 06
 * Tabla de compatibilidad de tipos
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION06_H
#define API_SESION06_H

typedef enum { TIPO_ERROR = 0, TIPO_CHAR = 1, TIPO_INT = 2, TIPO_FLOAT = 3 } Tipo;

int compatible(Tipo destino, Tipo origen);
const char *nombre_tipo(Tipo t);

#endif /* API_SESION06_H */


Writing api_sesion06.h


In [4]:
%%writefile artesano_sesion06_c.c

#include "api_sesion06.h"

int compatible(Tipo destino, Tipo origen) {

    /* Primero revisamos si alguno es un error */
    if (destino == TIPO_ERROR || origen == TIPO_ERROR) {
        return 0;
    }

    /* Comparamos los tipos que pueden ser compatibles */

    if (destino == TIPO_CHAR && origen == TIPO_CHAR) {
        return 1;
    }

    if (destino == TIPO_INT &&
        (origen == TIPO_CHAR || origen == TIPO_INT)) {
        return 1;
    }

    if (destino == TIPO_FLOAT &&
        (origen == TIPO_CHAR || origen == TIPO_INT ||
         origen == TIPO_FLOAT)) {
        return 1;
    }

    /* Si no cumple ninguna condicion, no es compatible */
    return 0;
}

const char *nombre_tipo(Tipo t) {

    /* Identificamos el nombre de cada tipo */
    if (t == TIPO_CHAR) {
        return "char";
    }

    if (t == TIPO_INT) {
        return "int";
    }

    if (t == TIPO_FLOAT) {
        return "float";
    }

    if (t == TIPO_ERROR) {
        return "error";
    }

    return "desconocido";
}


Overwriting artesano_sesion06_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion06_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


Contexto: estoy realizando una práctica de Lenguajes y Autómatas II en lenguaje C, necesito implementar una tabla de compatibilidad de tipos primitivos que permita identificar cuándo una asignación entre dos tipos de datos es correcta.

Rol: quiero que actúes como un programador que me ayude a resolver el ejercicio de manera clara, sencilla y fácil de comprender, utilizando conocimientos básicos de programación en C.

Especificación: Necesito implementar las funciones compatible(Tipo destino, Tipo origen) y nombre_tipo(Tipo t), respetando las declaraciones del archivo api_sesion06.h.

La jerarquía de los tipos es CHAR, INT y FLOAT. Se deben aceptar las asignaciones cuando el destino tenga el mismo rango o uno mayor que el origen. Cuando se intente asignar un tipo más grande a uno más pequeño, se debe rechazar. También se debe comprobar que TIPO_ERROR nunca sea compatible con otro tipo.

Quiero que esta segunda implementación sea diferente a la que realicé manualmente en la fase del Artesano, utilizando otra estructura de programación, pero que cumpla exactamente con las mismas condiciones.

Output: quiero recibir el código completo en C, sin incluir la función main(), respetando las firmas originales y evitando advertencias al compilar con -Wall -Wextra -Werror. También necesito que las seis pruebas del harness sean correctas y que el resultado final sea de 6/6 pruebas superadas.

In [9]:
%%writefile ia_sesion06_c.c
#include "api_sesion06.h"

int compatible(Tipo destino, Tipo origen) {

    if (destino == TIPO_ERROR || origen == TIPO_ERROR) {
        return 0;
    }

    if (destino < TIPO_CHAR || destino > TIPO_FLOAT ||
        origen < TIPO_CHAR || origen > TIPO_FLOAT) {
        return 0;
    }

    return destino >= origen;
}

const char *nombre_tipo(Tipo t) {

    switch (t) {
        case TIPO_ERROR:
            return "error";

        case TIPO_CHAR:
            return "char";

        case TIPO_INT:
            return "int";

        case TIPO_FLOAT:
            return "float";

        default:
            return "desconocido";
    }
}

Overwriting ia_sesion06_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [10]:
%%writefile harness_sesion06_c.c
/* harness_sesion06_c.c - auditoria automatica de la Sesion 06
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion06_c.c artesano_sesion06_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion06.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 06 - Tabla de compatibilidad de tipos ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    {  verificar_int("compatible(INT,INT)", (long)(1), (long)(compatible(TIPO_INT, TIPO_INT))); }
    {  verificar_int("compatible(FLOAT,INT)", (long)(1), (long)(compatible(TIPO_FLOAT, TIPO_INT))); }
    {  verificar_int("compatible(INT,FLOAT)", (long)(0), (long)(compatible(TIPO_INT, TIPO_FLOAT))); }
    {  verificar_int("compatible(INT,ERROR)", (long)(0), (long)(compatible(TIPO_INT, TIPO_ERROR))); }
    {  verificar_str("nombre_tipo(INT)", "int", nombre_tipo(TIPO_INT)); }
    {  verificar_str("nombre_tipo(ERROR)", "error", nombre_tipo(TIPO_ERROR)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion06.md" "\n");
    return 0;
}


Writing harness_sesion06_c.c


In [16]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion06_c.c artesano_sesion06_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 06 - Tabla de compatibilidad de tipos ===
Caso                               esperado:              obtenido:              resultado
[ 1] compatible(INT,INT)                esperado: 1            obtenido: 1            OK
[ 2] compatible(FLOAT,INT)              esperado: 1            obtenido: 1            OK
[ 3] compatible(INT,FLOAT)              esperado: 0            obtenido: 0            OK
[ 4] compatible(INT,ERROR)              esperado: 0            obtenido: 0            OK
[ 5] nombre_tipo(INT)                   esperado: int          obtenido: int          OK
[ 6] nombre_tipo(ERROR)                 esperado: error        obtenido: error        OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion06.md


In [14]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion06_c.c ia_sesion06_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 06 - Tabla de compatibilidad de tipos ===
Caso                               esperado:              obtenido:              resultado
[ 1] compatible(INT,INT)                esperado: 1            obtenido: 1            OK
[ 2] compatible(FLOAT,INT)              esperado: 1            obtenido: 1            OK
[ 3] compatible(INT,FLOAT)              esperado: 0            obtenido: 0            OK
[ 4] compatible(INT,ERROR)              esperado: 0            obtenido: 0            OK
[ 5] nombre_tipo(INT)                   esperado: int          obtenido: int          OK
[ 6] nombre_tipo(ERROR)                 esperado: error        obtenido: error        OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion06.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion06.md` a `auditorias/auditoria_sesion06.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
